# Toxic Comment Classifier — Training Walkthrough

This notebook walks through the full pipeline:
1. Data exploration
2. Tokenization
3. Model architecture overview
4. Training (or loading a pre-trained checkpoint)
5. Evaluation + per-label metrics
6. Live inference examples

**To run end-to-end:**
```bash
# Option A — use real Kaggle data
# Download from https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge
# Place train.csv in ./data/

# Option B — use synthetic data (runs in < 2 minutes on CPU)
python scripts/generate_sample_data.py
```

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch

from model.classifier import ToxicClassifier
from model.dataset import load_dataframes, make_loaders

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. Data Exploration

In [ ]:
LABELS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

train_df, val_df, test_df = load_dataframes('./data', sample_frac=1.0)
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
train_df.head()

In [ ]:
# Label distribution
label_counts = train_df[LABELS].sum().sort_values(ascending=False)
label_pcts = (label_counts / len(train_df) * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
label_counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Label counts (train set)')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

label_pcts.plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Label prevalence (%)')
axes[1].set_ylabel('% of comments')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print('Class imbalance ratio (clean:toxic):', round(len(train_df) / (train_df['toxic'].sum() + 1)))

In [ ]:
# Text length distribution
train_df['char_len'] = train_df['comment_text'].str.len()
train_df['word_count'] = train_df['comment_text'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_df['char_len'].clip(upper=2000).hist(bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Character length distribution (clipped at 2000)')
train_df['word_count'].clip(upper=300).hist(bins=50, ax=axes[1], color='teal')
axes[1].set_title('Word count distribution (clipped at 300)')
plt.tight_layout()
plt.show()

print(f'Median word count: {train_df["word_count"].median():.0f}')
print(f'95th percentile word count: {train_df["word_count"].quantile(0.95):.0f}')
print('Note: max_length=128 tokens covers the majority of comments.')

## 2. Tokenization

In [ ]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

example = train_df['comment_text'].iloc[0]
tokens = tokenizer(example, truncation=True, max_length=128)
print('Example comment:')
print(example[:200], '...')
print(f'\nToken count: {len(tokens["input_ids"])}')
print('First 20 tokens:', tokenizer.convert_ids_to_tokens(tokens['input_ids'][:20]))

## 3. Model Architecture

In [ ]:
model = ToxicClassifier()
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Architecture: DistilBERT → [CLS] → Dropout(0.3) → Linear({model.bert.config.hidden_size}, {model.num_labels}) → Sigmoid')
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Labels: {ToxicClassifier.LABELS}')

# Quick forward pass
dummy_ids = torch.randint(0, 1000, (2, 128))
dummy_mask = torch.ones(2, 128, dtype=torch.long)
with torch.no_grad():
    out = model(dummy_ids, dummy_mask)
print(f'\nOutput shape: {out.shape}  (batch=2, labels=6)')
print(f'Output range: [{out.min():.4f}, {out.max():.4f}]  (sigmoid → [0,1])')

## 4. Training

Run from the command line for full training:
```bash
python -m model.train --data-dir ./data --epochs 3 --batch-size 32
```

Or trigger a short run below (1 epoch, 10% data):

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '-m', 'model.train', '--data-dir', './data', '--epochs', '1', '--sample-frac', '0.1'],
    capture_output=True, text=True, cwd='..'
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

## 5. Evaluation

In [ ]:
import json
from pathlib import Path

results_path = Path('../checkpoints/training_results.json')
if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)

    print(f'Best val AUC: {results["best_val_auc"]:.4f}')
    print(f'Test AUC:     {results["test_auc"]:.4f}')

    history = results['history']
    epochs = [h['epoch'] for h in history]
    val_auc = [h['val_auc'] for h in history]
    train_loss = [h['train_loss'] for h in history]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, train_loss, 'o-', label='Train loss')
    axes[0].set_title('Training Loss')
    axes[0].set_xlabel('Epoch')
    axes[1].plot(epochs, val_auc, 's-', color='coral', label='Val AUC')
    axes[1].set_title('Validation ROC-AUC')
    axes[1].set_xlabel('Epoch')
    plt.tight_layout()
    plt.show()
else:
    print('No training results found. Run training first.')

## 6. Live Inference

In [ ]:
from pathlib import Path

ckpt = Path('../checkpoints/best_model.pt')
if ckpt.exists():
    from model.predict import ToxicPredictor
    predictor = ToxicPredictor(str(ckpt), device='cpu')

    test_comments = [
        'I really enjoyed reading this, thank you for sharing.',
        'You are completely wrong and an absolute idiot.',
        'Can anyone explain how to set up a Python virtual environment?',
        'I hate people like you, you should disappear.',
    ]

    for comment in test_comments:
        result = predictor.predict_with_explanation(comment)
        print(f'Text: "{comment[:60]}..."' if len(comment) > 60 else f'Text: "{comment}"')
        print(f'  → {result["summary"]} (top score: {result["top_score"]:.3f})')
        active = {k: round(v, 3) for k, v in result['scores'].items() if v > 0.1}
        if active:
            print(f'  → Scores: {active}')
        print()
else:
    print('Train the model first: python -m model.train --data-dir ./data')